# Production Model: Hyperparameter Optimization

**Goal**: Find the best model + hyperparameters for lab deployment by evaluating on the existing OOD cluster splits.

**Approach**:
1. Define hyperparameter grids for RF (12 combos), HistGradientBoosting (6), and MLP (6)
2. For each combo, train on train clusters and evaluate on test clusters across 3 iterations
3. Average OOD performance selects the best configuration per model
4. Compare the 3 optimized models, pick the best
5. Retrain the winner on the full dataset and save for production

**Feature sets tested**: DRFP (2048), split (substrate DRFP 2048 + reagent QM 21 = 2069)

## 1. Setup & Data Loading

In [1]:
import pickle
import warnings
import copy
import sys
import types

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime
from itertools import product as iter_product

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

from utils.ml_train import get_test_sets_per_iteration
from utils.performance import calculate_metrics

# Pandas 2.x compatibility
if "pandas.core.indexes.numeric" not in sys.modules:
    numeric_module = types.ModuleType("pandas.core.indexes.numeric")
    sys.modules["pandas.core.indexes.numeric"] = numeric_module
else:
    numeric_module = sys.modules["pandas.core.indexes.numeric"]
try:
    from pandas.core.indexes.base import Index as _PandasIndex
    for _cls_name in ("NumericIndex", "Int64Index", "UInt64Index", "Float64Index"):
        if not hasattr(numeric_module, _cls_name):
            setattr(numeric_module, _cls_name, _PandasIndex)
except Exception:
    pass

warnings.filterwarnings("ignore")

year = datetime.now().year
month = datetime.now().month
day = datetime.now().day

In [11]:
pkl_path = "/projects/reactions/HTE_Data_Production/Buchwald/Manuscript Code/data/2026-02-21_df_HTE_BH_QM_10iter_9-3clustertest.pkl"

with open(pkl_path, "rb") as f:
    df_HTE_BH = pickle.load(f)

In [16]:
len(df_HTE_BH)

27497

### Load dataset and compute features

In [2]:
pkl_path = "/projects/reactions/HTE_Data_Production/Buchwald/Manuscript Code/data/2026-02-21_df_HTE_BH_QM_10iter_9-3clustertest.pkl"

with open(pkl_path, "rb") as f:
    df_HTE_BH = pickle.load(f)

print(f"Loaded {len(df_HTE_BH)} reactions")
print(f"Sources: {df_HTE_BH['Source'].value_counts().to_string()}")
print(f"Labels: {df_HTE_BH['labels'].value_counts().to_dict()}")
print(f"\nDRFP shape: {df_HTE_BH['DRFP'].iloc[0].shape}")
print(f"QM shape: {df_HTE_BH['QM'].iloc[0].shape}")

Loaded 27497 reactions
Sources: Source
JNJ HTE 2024         11328
Dreher Doyle 2018     4312
Merck 2025            4204
Roche 2023            3359
HiTEA 2024            2632
Dreher 2014            768
AZ ELN 2023            750
Dreher 2016            144
Labels: {0: 15532, 1: 11965}

DRFP shape: (2048,)
QM shape: (50,)


### Compute split features: DRFP_subst_prod + reagent QM

- **DRFP_subst_prod** (2048): pre-computed in pickle — XOR(product_fp, aryl_fp | amine_fp), excludes reagents
- **Reagent QM** (21): QM descriptors for catalyst (7) + base (7) + solvent (7) — indices 14-34 of QM array

In [3]:
# Extract reagent QM (catalyst + base + solvent = indices 14-34 of QM array)
print("Extracting reagent QM features...")
df_HTE_BH["reagent_QM"] = df_HTE_BH["QM"].apply(lambda qm: qm[14:35])

# Combine: DRFP_subst_prod (2048) + reagent QM (21) = 2069
df_HTE_BH["split_features"] = df_HTE_BH.apply(
    lambda row: np.concatenate([row["DRFP_subst_prod"].astype(np.float64), row["reagent_QM"]]),
    axis=1,
)

# Clean NaN/inf
nan_count = sum(np.any(~np.isfinite(v)) for v in df_HTE_BH["split_features"])
if nan_count > 0:
    print(f"Cleaning {nan_count} rows with NaN/inf")
    df_HTE_BH["split_features"] = df_HTE_BH["split_features"].apply(
        lambda x: np.where(np.isfinite(x), x, 0.0)
    )

print(f"\nFeature sets available:")
print(f"  DRFP:            {df_HTE_BH['DRFP'].iloc[0].shape}")
print(f"  QM:              {df_HTE_BH['QM'].iloc[0].shape}")
print(f"  DRFP_subst_prod: {df_HTE_BH['DRFP_subst_prod'].iloc[0].shape}")
print(f"  split_features:  {df_HTE_BH['split_features'].iloc[0].shape} (DRFP_subst_prod 2048 + reagent_QM 21)")

Extracting reagent QM features...

Feature sets available:
  DRFP:            (2048,)
  QM:              (50,)
  DRFP_subst_prod: (2048,)
  split_features:  (2069,) (DRFP_subst_prod 2048 + reagent_QM 21)


## 2. Hyperparameter Grids

RF: 12 combos (fast). HGB and MLP: 6 combos each (slower to train). Each evaluated across 3 OOD iterations.

In [4]:
# --- RF grid: 12 combos (fast, keep full grid) ---
rf_grid = []
rf_fixed = dict(max_depth=None, min_samples_split=4, criterion="entropy",
                class_weight="balanced", random_state=42, n_jobs=-1)
for n_est in [200, 300, 500]:
    for leaf in [1, 3]:
        for feat in ["log2", "sqrt"]:
            rf_grid.append({**rf_fixed, "n_estimators": n_est,
                           "min_samples_leaf": leaf, "max_features": feat})

# --- HistGradientBoosting grid: 6 combos (slow, reduced) ---
# Fix max_depth=6; vary max_iter and learning_rate
hgb_grid = []
for max_iter in [200, 400]:
    for lr in [0.05, 0.1, 0.2]:
        hgb_grid.append(dict(max_iter=max_iter, max_depth=6,
                            learning_rate=lr, random_state=42))

# --- MLP grid: 6 combos (slow, reduced) ---
mlp_grid = []
for hidden in [(64, 16), (100, 20), (200, 50)]:
    for alpha in [1e-3, 1e-4]:
        mlp_grid.append(dict(hidden_layer_sizes=hidden, alpha=alpha,
                            solver="lbfgs", max_iter=300, random_state=42))

print(f"RF:  {len(rf_grid)} combos")
print(f"HGB: {len(hgb_grid)} combos")
print(f"MLP: {len(mlp_grid)} combos")

# Feature sets to test (DRFP and split only; QM dropped for runtime)
feature_sets = {
    "DRFP": "DRFP",
    "split": "split_features",
}
print(f"\nFeature sets: {list(feature_sets.keys())}")
total_configs = (len(rf_grid) + len(hgb_grid) + len(mlp_grid)) * len(feature_sets)
print(f"Total configs: {total_configs}")

RF:  12 combos
HGB: 6 combos
MLP: 6 combos

Feature sets: ['DRFP', 'split']
Total configs: 48


## 3. Grid Search on OOD Splits

For each hyperparameter combo: train on train clusters, evaluate on test clusters, average across 3 iterations.

In [5]:
def grid_search_ood(df, model_class, param_grid, feat_col, n_iterations=10):
    """
    Evaluate each hyperparameter combo across OOD cluster splits.
    
    For each combo, trains on train clusters and evaluates on test clusters
    across all iterations. Returns a DataFrame with averaged metrics per combo.
    """
    test_sets_all = get_test_sets_per_iteration(df)
    metrics_names = ['ROC AUC', 'Balanced Accuracy', 'F1 Score', 'F0 Score',
                     'AU-PR-C', 'Precision 1', 'Recall 1']
    results = []

    for pidx, params in enumerate(tqdm(param_grid, desc=f"{model_class.__name__}")):
        raw = {m: [] for m in metrics_names}

        for rand_state in range(n_iterations):
            cluster_col = f"iteration_{rand_state} cluster"
            test_sets = test_sets_all[rand_state]

            for test_clusters in test_sets:
                df_train = df[~df[cluster_col].isin(test_clusters)]
                df_test = df[df[cluster_col].isin(test_clusters)]

                if len(df_test) == 0:
                    continue

                X_train = np.vstack(df_train[feat_col].values)
                y_train = df_train["labels"].values

                scaler = StandardScaler()
                X_train_sc = scaler.fit_transform(X_train)

                model = model_class(**params)
                model.fit(X_train_sc, y_train)

                models_dict = {model_class.__name__: model}
                metrics, _ = calculate_metrics(models_dict, scaler, df_test, feat_col)

                for m in metrics_names:
                    raw[m].append(metrics[model_class.__name__][m])

        row = {}
        # Store hyperparameters (only the varying ones)
        for k, v in params.items():
            if k not in ("random_state", "n_jobs"):
                row[k] = v
        # Store averaged metrics
        for m in metrics_names:
            vals = np.array(raw[m], dtype=np.float64)
            row[f"{m} Avg"] = np.nanmean(vals)
            row[f"{m} Std"] = np.nanstd(vals)
        row["n_splits"] = len(raw[metrics_names[0]])
        results.append(row)

    return pd.DataFrame(results)

print("grid_search_ood defined.")

grid_search_ood defined.


In [6]:
# --- Run grid search for all models × all feature sets ---
N_ITERATIONS = 3  # reduced from 10 for runtime

model_grids = {
    "RF": (RandomForestClassifier, rf_grid),
    "HGB": (HistGradientBoostingClassifier, hgb_grid),
    "MLP": (MLPClassifier, mlp_grid),
}

all_grid_results = {}
ranking_metric = "ROC AUC Avg"

# Count total HP combos across all feature sets and models
total_combos = sum(len(grid) for _, grid in model_grids.values()) * len(feature_sets)
overall_pbar = tqdm(total=total_combos, desc="Overall grid search")

for feat_name, feat_col in feature_sets.items():
    print(f"\n{'='*70}")
    print(f"Feature set: {feat_name} ({feat_col})")
    print(f"{'='*70}")

    for model_name, (model_class, param_grid) in model_grids.items():
        print(f"\n--- {model_name} ({len(param_grid)} combos) ---")
        df_grid = grid_search_ood(df_HTE_BH, model_class, param_grid,
                                  feat_col=feat_col, n_iterations=N_ITERATIONS)
        df_grid = df_grid.sort_values(ranking_metric, ascending=False).reset_index(drop=True)

        key = f"{feat_name}_{model_name}"
        all_grid_results[key] = df_grid

        # Print top 3
        best = df_grid.iloc[0]
        print(f"  Best: ROC={best['ROC AUC Avg']:.4f}, BA={best['Balanced Accuracy Avg']:.4f}, "
              f"F1={best['F1 Score Avg']:.4f}")

        # Save per-model grid results
        csv_path = f"results/GridSearch_{feat_name}_{model_name}_{year}-{month}-{day}.csv"
        df_grid.to_csv(csv_path, index=False)

        overall_pbar.update(len(param_grid))

overall_pbar.close()
print(f"\n{'='*70}")
print("Grid search complete.")
print(f"{'='*70}")

Overall grid search:   0%|          | 0/48 [00:00<?, ?it/s]


Feature set: DRFP (DRFP)

--- RF (12 combos) ---


RandomForestClassifier:   0%|          | 0/12 [00:00<?, ?it/s]

  Best: ROC=0.8112, BA=0.7090, F1=0.5927

--- HGB (6 combos) ---


HistGradientBoostingClassifier:   0%|          | 0/6 [00:00<?, ?it/s]

...

  Best: ROC=0.8474, BA=0.7668, F1=0.7354

--- HGB (6 combos) ---


HistGradientBoostingClassifier:   0%|          | 0/6 [00:00<?, ?it/s]

  Best: ROC=0.8249, BA=0.7371, F1=0.6873

--- MLP (6 combos) ---


MLPClassifier:   0%|          | 0/6 [00:00<?, ?it/s]

  Best: ROC=0.7909, BA=0.7408, F1=0.7151

Grid search complete.


## 4. Results Summary

Best hyperparameter configuration per model per feature set, ranked by ROC AUC.

In [7]:
# --- Best configuration per model × feature set ---
summary_rows = []

for feat_name in feature_sets:
    for model_name in model_grids:
        key = f"{feat_name}_{model_name}"
        df_grid = all_grid_results[key]
        best = df_grid.iloc[0]
        summary_rows.append({
            "Feature Set": feat_name,
            "Model": model_name,
            "ROC AUC": best["ROC AUC Avg"],
            "ROC Std": best["ROC AUC Std"],
            "Balanced Acc": best["Balanced Accuracy Avg"],
            "BA Std": best["Balanced Accuracy Std"],
            "F1 Score": best["F1 Score Avg"],
            "F1 Std": best["F1 Score Std"],
            "Precision": best.get("Precision 1 Avg", np.nan),
            "Recall": best.get("Recall 1 Avg", np.nan),
        })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values("ROC AUC", ascending=False).reset_index(drop=True)

print("Best configuration per model × feature set (ranked by ROC AUC):\n")
print(df_summary.to_string(index=False))

# Overall best
best_overall = df_summary.iloc[0]
print(f"\n{'='*70}")
print(f"OVERALL BEST: {best_overall['Model']} + {best_overall['Feature Set']}")
print(f"  ROC AUC = {best_overall['ROC AUC']:.4f} ± {best_overall['ROC Std']:.4f}")
print(f"  BA      = {best_overall['Balanced Acc']:.4f} ± {best_overall['BA Std']:.4f}")
print(f"  F1      = {best_overall['F1 Score']:.4f} ± {best_overall['F1 Std']:.4f}")
print(f"{'='*70}")

# Save summary
df_summary.to_csv(f"results/Production_HP_Summary_{year}-{month}-{day}.csv", index=False)

Best configuration per model × feature set (ranked by ROC AUC):

Feature Set Model  ROC AUC  ROC Std  Balanced Acc   BA Std  F1 Score   F1 Std  Precision   Recall
      split    RF 0.847417 0.054047      0.766799 0.061827  0.735365 0.089092   0.758401 0.722449
      split   HGB 0.824852 0.072064      0.737062 0.071852  0.687269 0.118298   0.757166 0.643254
       DRFP    RF 0.811169 0.070568      0.709040 0.093729  0.592740 0.202068   0.824355 0.497840
       DRFP   HGB 0.808073 0.079284      0.718126 0.071721  0.638232 0.139255   0.801084 0.548067
      split   MLP 0.790927 0.090826      0.740768 0.066202  0.715120 0.076780   0.700333 0.737545
       DRFP   MLP 0.727427 0.080069      0.671386 0.069026  0.609105 0.121389   0.665716 0.590185

OVERALL BEST: RF + split
  ROC AUC = 0.8474 ± 0.0540
  BA      = 0.7668 ± 0.0618
  F1      = 0.7354 ± 0.0891


## 5. Print Best Hyperparameters per Model

Show the winning hyperparameter configuration for each model + feature set combination.

In [8]:
# --- Print best hyperparameters for each model × feature set ---
best_configs = {}

for feat_name in feature_sets:
    for model_name, (model_class, param_grid) in model_grids.items():
        key = f"{feat_name}_{model_name}"
        df_grid = all_grid_results[key]
        best_row = df_grid.iloc[0]

        # Extract HP columns (everything that's not a metric)
        metric_cols = [c for c in df_grid.columns if "Avg" in c or "Std" in c or c == "n_splits"]
        hp_cols = [c for c in df_grid.columns if c not in metric_cols]
        hp_dict = {c: best_row[c] for c in hp_cols}

        best_configs[key] = {"model_class": model_class, "params": hp_dict,
                             "feat_col": feature_sets[feat_name]}

        print(f"\n{feat_name} / {model_name}:")
        print(f"  ROC={best_row['ROC AUC Avg']:.4f}, BA={best_row['Balanced Accuracy Avg']:.4f}, "
              f"F1={best_row['F1 Score Avg']:.4f}")
        for k, v in hp_dict.items():
            print(f"    {k}: {v}")


DRFP / RF:
  ROC=0.8112, BA=0.7090, F1=0.5927
    max_depth: None
    min_samples_split: 4
    criterion: entropy
    class_weight: balanced
    n_estimators: 300
    min_samples_leaf: 1
    max_features: log2

DRFP / HGB:
  ROC=0.8081, BA=0.7181, F1=0.6382
    max_iter: 200.0
    max_depth: 6.0
    learning_rate: 0.1

DRFP / MLP:
  ROC=0.7274, BA=0.6714, F1=0.6091
    hidden_layer_sizes: (200, 50)
    alpha: 0.0001
    solver: lbfgs
    max_iter: 300

split / RF:
  ROC=0.8474, BA=0.7668, F1=0.7354
    max_depth: None
    min_samples_split: 4
    criterion: entropy
    class_weight: balanced
    n_estimators: 500
    min_samples_leaf: 1
    max_features: log2

split / HGB:
  ROC=0.8249, BA=0.7371, F1=0.6873
    max_iter: 400.0
    max_depth: 6.0
    learning_rate: 0.2

split / MLP:
  ROC=0.7909, BA=0.7408, F1=0.7151
    hidden_layer_sizes: (100, 20)
    alpha: 0.001
    solver: lbfgs
    max_iter: 300


## 6. Train Production Model on Full Dataset

Train the overall best model + feature set on ALL available data and save for lab deployment.

In [9]:
# --- Identify overall best model + feature set ---
best_key = df_summary.iloc[0]["Model"] + " + " + df_summary.iloc[0]["Feature Set"]
best_feat_name = df_summary.iloc[0]["Feature Set"]
best_model_name = df_summary.iloc[0]["Model"]

# Get the config
config_key = f"{best_feat_name}_{best_model_name}"
best_cfg = best_configs[config_key]
best_model_class = best_cfg["model_class"]
best_feat_col = best_cfg["feat_col"]

# Reconstruct full params (add back fixed params like random_state, n_jobs)
best_params = dict(best_cfg["params"])
best_params["random_state"] = 42
if best_model_class == RandomForestClassifier:
    best_params["n_jobs"] = -1

print(f"Training production model: {best_model_name} on {best_feat_name}")
print(f"Feature column: {best_feat_col}")
print(f"Hyperparameters: {best_params}")

# Train on full dataset
X_all = np.vstack(df_HTE_BH[best_feat_col].values)
y_all = df_HTE_BH["labels"].values

print(f"\nTraining on {X_all.shape[0]} reactions, {X_all.shape[1]} features")
print(f"Label distribution: 0={np.sum(y_all == 0)}, 1={np.sum(y_all == 1)}")

prod_scaler = StandardScaler()
X_all_sc = prod_scaler.fit_transform(X_all)

prod_model = best_model_class(**best_params)
prod_model.fit(X_all_sc, y_all)

print(f"\nProduction model trained successfully.")

Training production model: RF on split
Feature column: split_features
Hyperparameters: {'max_depth': None, 'min_samples_split': np.int64(4), 'criterion': 'entropy', 'class_weight': 'balanced', 'n_estimators': np.int64(500), 'min_samples_leaf': np.int64(1), 'max_features': 'log2', 'random_state': 42, 'n_jobs': -1}

Training on 27497 reactions, 2069 features
Label distribution: 0=15532, 1=11965

Production model trained successfully.


In [10]:
# --- Save production model artifact ---
model_artifact = {
    "model": prod_model,
    "scaler": prod_scaler,
    "feature_set": best_feat_name,
    "feature_column": best_feat_col,
    "model_type": best_model_name,
    "hyperparameters": best_params,
    "training_samples": X_all.shape[0],
    "feature_dim": X_all.shape[1],
    "ood_metrics": {
        "ROC AUC": float(df_summary.iloc[0]["ROC AUC"]),
        "Balanced Accuracy": float(df_summary.iloc[0]["Balanced Acc"]),
        "F1 Score": float(df_summary.iloc[0]["F1 Score"]),
    },
    "date": f"{year}-{month}-{day}",
}

model_path = f"model/production_model_{best_feat_name}_{best_model_name}_{year}-{month}-{day}.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model_artifact, f)

print(f"Production model saved to {model_path}")
print(f"\nArtifact contents:")
for k, v in model_artifact.items():
    if k not in ("model", "scaler"):
        print(f"  {k}: {v}")

Production model saved to model/production_model_split_RF_2026-2-24.pkl

Artifact contents:
  feature_set: split
  feature_column: split_features
  model_type: RF
  hyperparameters: {'max_depth': None, 'min_samples_split': np.int64(4), 'criterion': 'entropy', 'class_weight': 'balanced', 'n_estimators': np.int64(500), 'min_samples_leaf': np.int64(1), 'max_features': 'log2', 'random_state': 42, 'n_jobs': -1}
  training_samples: 27497
  feature_dim: 2069
  ood_metrics: {'ROC AUC': 0.8474173495423273, 'Balanced Accuracy': 0.7667993060743976, 'F1 Score': 0.7353651588666552}
  date: 2026-2-24


### Usage: loading the production model for inference

```python
import pickle
import numpy as np

with open("model/production_model_split_RF_2026-2-24.pkl", "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]
scaler = artifact["scaler"]

# X_new: array of shape (n_reactions, 2069) — DRFP_subst_prod (2048) + reagent QM (21)
X_new_scaled = scaler.transform(X_new)
predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)[:, 1]
```